##Recall Postcell Class

In [ ]:
#!pip install postcell

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%cd /content/drive/MyDrive/2026/Markoding/Week-1-pi-integrasi-ai.git #silahkan ganti dengan lokasi hasil clone kalian

!git status

In [ ]:
%reload_ext postcell
%postcell register

# Hands-on: AI Recommendation System dengan TF-IDF
**Studi kasus:** rekomendasi produk ramah lingkungan berdasarkan preferensi pengguna  
**Platform:** Google Colab | **Level:** Pemula | **Estimasi:** 45–60 menit

## Tujuan pembelajaran
Setelah menyelesaikan notebook ini, peserta dapat:
1. Menjelaskan bagaimana TF-IDF mengubah teks menjadi representasi numerik.
2. Membuat fitur teks menggunakan `TfidfVectorizer` dari scikit-learn.
3. Menghitung cosine similarity antara preferensi pengguna dan deskripsi produk.
4. Membuat fungsi rekomendasi Top-K berdasarkan skor kemiripan.
5. Menguji perubahan hasil rekomendasi dan mendiskusikan keterbatasan TF-IDF.

> **Catatan pengajar:** Pada versi ini, kita menggunakan pendekatan content-based recommendation berbasis TF-IDF. TF-IDF tidak memahami makna seperti manusia; metode ini memberi bobot pada kata berdasarkan kemunculan kata dalam dokumen dan seluruh katalog.


## 1. Gambaran alur sistem

1. **Input pengguna** — misalnya, “Saya ingin mengurangi penggunaan plastik.”
2. **Katalog produk** — setiap item memiliki nama dan deskripsi.
3. **TF-IDF vectorizer** — mengubah deskripsi dan preferensi menjadi vektor berbobot kata.
4. **Cosine similarity** — membandingkan vektor preferensi dengan vektor produk.
5. **Ranking** — menampilkan produk dengan skor tertinggi.

**Istilah penting**
- **TF (Term Frequency):** seberapa sering suatu kata muncul dalam sebuah dokumen.
- **IDF (Inverse Document Frequency):** bobot yang cenderung lebih tinggi untuk kata yang lebih jarang muncul di seluruh kumpulan dokumen.
- **TF-IDF:** representasi teks berbasis bobot kata; biasanya menghasilkan vektor sparse.
- **Cosine similarity:** mengukur kesamaan arah dua vektor.
- **Top-K:** sejumlah K rekomendasi teratas.

**Batasan:** contoh ini adalah *content-based recommendation*, bukan collaborative filtering. Sistem tidak mempelajari pola pembelian atau perilaku pengguna lain.

**Pertanyaan utama:** jika preferensi pengguna menggunakan kata yang berbeda dari deskripsi produk, apakah TF-IDF tetap bisa menemukan item yang tepat?


## 2. Instalasi library
Notebook ini menggunakan `pandas` untuk mengelola katalog serta `scikit-learn` untuk TF-IDF dan cosine similarity. Google Colab biasanya sudah menyediakan library tersebut.

In [2]:
%pip -q install pandas scikit-learn

## 3. Import library
- `TfidfVectorizer` membangun vocabulary dan bobot TF-IDF dari deskripsi produk.
- `cosine_similarity` menghitung kemiripan antara vektor query dan vektor produk.

In [3]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## 4. Membuat dataset mini
Dataset produk berikut **sengaja dipertahankan sama persis** dengan notebook pretrained-model agar hasil kedua pendekatan dapat dibandingkan pada use case dan data yang sama. Dataset ini dibuat langsung di notebook agar peserta tidak perlu mengunduh file eksternal.

In [4]:
products = pd.DataFrame([
    {
        "nama": "Botol minum reusable",
        "kategori": "Perlengkapan",
        "deskripsi": "Botol minum yang dapat digunakan berulang kali untuk mengurangi sampah plastik"
    },
    {
        "nama": "Tas belanja kain",
        "kategori": "Perlengkapan",
        "deskripsi": "Tas kain yang dapat digunakan kembali saat berbelanja tanpa kantong plastik"
    },
    {
        "nama": "Sedotan bambu",
        "kategori": "Perlengkapan",
        "deskripsi": "Sedotan ramah lingkungan yang bisa digunakan kembali"
    },
    {
        "nama": "Lampu tenaga surya",
        "kategori": "Energi",
        "deskripsi": "Lampu hemat energi yang menggunakan tenaga matahari"
    },
    {
        "nama": "Komposter rumah tangga",
        "kategori": "Pengelolaan sampah",
        "deskripsi": "Alat untuk mengolah sampah organik menjadi kompos"
    },
    {
        "nama": "Kotak makan reusable",
        "kategori": "Perlengkapan",
        "deskripsi": "Wadah makanan yang dapat digunakan kembali untuk mengurangi sampah kemasan"
    }
])

products

,nama,kategori,deskripsi
0,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...
1,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...
2,Sedotan bambu,Perlengkapan,Sedotan ramah lingkungan yang bisa digunakan k...
3,Lampu tenaga surya,Energi,Lampu hemat energi yang menggunakan tenaga mat...
4,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos
5,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...


### Catatan diskusi
- Kata apa yang mungkin membantu mencocokkan preferensi dengan produk?
- Bagaimana hasilnya jika pengguna memakai sinonim yang tidak muncul dalam deskripsi?
- Apakah enam produk ini cukup untuk menyimpulkan kualitas sistem pada penggunaan nyata?


## 5. Membuat representasi TF-IDF
Kita menggunakan `TfidfVectorizer` untuk membangun vocabulary dari deskripsi produk dan mengubah setiap deskripsi menjadi vektor berbobot kata.

Secara intuitif:

**TF-IDF = bobot kemunculan kata dalam dokumen × bobot kelangkaan kata di seluruh katalog**

Scikit-learn menangani detail perhitungan dan normalisasi. Hasilnya biasanya berupa *sparse matrix*: nilai nol tidak disimpan secara eksplisit agar penggunaan memori lebih efisien.

Catatan: vocabulary di-*fit* pada katalog produk, lalu preferensi pengguna diubah menggunakan vocabulary yang sama. Jangan melakukan `fit_transform` ulang pada query pengguna.

In [5]:
# Kita memakai kolom deskripsi yang sama untuk seluruh produk.
# stop_words tidak ditetapkan agar contoh bahasa Indonesia tetap sederhana
# dan tidak bergantung pada daftar stop words bahasa Inggris.
vectorizer = TfidfVectorizer(lowercase=True)

product_tfidf = vectorizer.fit_transform(products["deskripsi"])

print("Jumlah produk:", product_tfidf.shape[0])
print("Jumlah fitur/kata dalam vocabulary:", product_tfidf.shape[1])
print("Bentuk matriks TF-IDF:", product_tfidf.shape)
print("Contoh vocabulary:", vectorizer.get_feature_names_out()[:20])

Jumlah produk: 6
Jumlah fitur/kata dalam vocabulary: 36
Bentuk matriks TF-IDF: (6, 36)
Contoh vocabulary: ['alat' 'berbelanja' 'berulang' 'bisa' 'botol' 'dapat' 'digunakan'
 'energi' 'hemat' 'kain' 'kali' 'kantong' 'kemasan' 'kembali' 'kompos'
 'lampu' 'lingkungan' 'makanan' 'matahari' 'menggunakan']


## 6. Memahami hasil TF-IDF
Setiap baris pada `product_tfidf` mewakili satu produk dan setiap kolom mewakili kata dalam vocabulary.

- **Baris:** produk.
- **Kolom:** kata/fitur teks.
- **Nilai:** bobot TF-IDF kata tersebut pada produk.
- **Nilai nol:** kata itu tidak muncul atau tidak memiliki bobot pada dokumen tersebut.

TF-IDF tidak menghasilkan embedding semantik seperti Transformer. Kata yang berbeda tetapi bermakna serupa, misalnya “hemat” dan “efisien”, tidak otomatis dianggap mirip kecuali keduanya memiliki overlap fitur yang mendukung.

In [6]:
# Tampilkan sebagian matriks sebagai tabel agar mudah dipahami.
# Untuk dataset besar, jangan mengubah seluruh sparse matrix menjadi dense.
feature_names = vectorizer.get_feature_names_out()
tfidf_preview = pd.DataFrame(
    product_tfidf[:3, :].toarray(),
    columns=feature_names
)
tfidf_preview.insert(0, "produk", products["nama"].iloc[:3].to_list())
tfidf_preview

,produk,alat,berbelanja,berulang,bisa,botol,dapat,digunakan,energi,hemat,...,ramah,saat,sampah,sedotan,tanpa,tas,tenaga,untuk,wadah,yang
0,Botol minum reusable,0.0,0.000000,0.367678,0.00000,0.367678,0.254548,0.218128,0.0,0.0,...,0.00000,0.000000,0.254548,0.00000,0.000000,0.000000,0.0,0.254548,0.0,0.188371
1,Tas belanja kain,0.0,0.348251,0.000000,0.00000,0.000000,0.241099,0.206603,0.0,0.0,...,0.00000,0.348251,0.000000,0.00000,0.348251,0.348251,0.0,0.000000,0.0,0.178418
2,Sedotan bambu,0.0,0.000000,0.000000,0.44308,0.000000,0.000000,0.262861,0.0,0.0,...,0.44308,0.000000,0.000000,0.44308,0.000000,0.000000,0.0,0.000000,0.0,0.227002


**Cara membaca output:** setiap baris adalah produk dan setiap kolom adalah kata dari vocabulary. Angka yang lebih besar menunjukkan kata tersebut lebih berbobot untuk dokumen itu menurut perhitungan TF-IDF. Nilai ini bukan skor relevansi terhadap pengguna; skor rekomendasi baru dihitung saat query pengguna dibandingkan dengan vektor produk.

## 7. Membuat fungsi rekomendasi
Fungsi berikut menerima preferensi dalam bentuk teks, mengubahnya menjadi vektor TF-IDF menggunakan vocabulary yang sudah dibangun, menghitung cosine similarity terhadap semua produk, lalu mengembalikan Top-K produk dengan skor tertinggi.

In [7]:
def recommend_products(preference, top_k=3):
    # Validasi input sederhana
    if not isinstance(preference, str) or not preference.strip():
        raise ValueError("Preferensi harus berupa teks yang tidak kosong.")

    if top_k < 1:
        raise ValueError("top_k harus minimal 1.")

    # Gunakan vocabulary yang sama dengan deskripsi produk
    query_tfidf = vectorizer.transform([preference])

    # Skor kemiripan query dengan seluruh produk
    scores = cosine_similarity(
        query_tfidf,
        product_tfidf
    )[0]

    # Ambil indeks dengan skor tertinggi
    ranked_indices = scores.argsort()[::-1][:min(top_k, len(products))]

    results = products.iloc[ranked_indices].copy()
    results["skor_kemiripan"] = scores[ranked_indices].round(3)

    return results.reset_index(drop=True)

### Memahami kode
- `vectorizer.transform([preference])`: mengubah preferensi pengguna menjadi vektor TF-IDF dengan vocabulary yang sudah ada.
- `cosine_similarity(...)`: menghasilkan skor antara query dan setiap produk.
- `argsort()[::-1]`: mengurutkan indeks dari skor tertinggi ke terendah.
- `top_k`: membatasi jumlah rekomendasi yang ditampilkan.

**Penting:** skor kemiripan bukan probabilitas dan bukan akurasi. Skor 0.70 tidak berarti peluang pengguna menyukai produk adalah 70%.

**Perhatikan juga:** jika query tidak memiliki kata yang cocok dengan vocabulary katalog, vektor query bisa seluruhnya nol sehingga skor semua produk menjadi nol. Ini salah satu keterbatasan pendekatan berbasis kata.

## 8. Uji rekomendasi pertama
Gunakan preferensi yang sama persis dengan notebook pretrained-model agar kedua metode dapat dibandingkan secara adil.

In [8]:
user_preference = (
    "Saya ingin mengurangi penggunaan plastik "
    "ketika membawa minuman dan makanan."
)

recommend_products(user_preference, top_k=3)

,nama,kategori,deskripsi,skor_kemiripan
0,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.438
1,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.323
2,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...,0.153


Latihan sendiri disini dengan menggunakan prompt kamu

## 9. Eksperimen: ubah preferensi pengguna
Coba masukan berikut. Query dipertahankan sama persis dengan notebook pretrained-model untuk memudahkan perbandingan hasil.

In [9]:
test_preferences = [
    "Saya ingin mengurangi sampah plastik saat berbelanja.",
    "Saya ingin menghemat energi di rumah.",
    "Saya ingin mengolah sampah makanan menjadi sesuatu yang berguna.",
    "Saya membutuhkan barang yang bisa dipakai berulang kali."
]

for preference in test_preferences:
    print("\nPreferensi:", preference)
    display(recommend_products(preference, top_k=3))


Preferensi: Saya ingin mengurangi sampah plastik saat berbelanja.


,nama,kategori,deskripsi,skor_kemiripan
0,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...,0.476
1,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.343
2,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.236



Preferensi: Saya ingin menghemat energi di rumah.


,nama,kategori,deskripsi,skor_kemiripan
0,Lampu tenaga surya,Energi,Lampu hemat energi yang menggunakan tenaga mat...,0.4
1,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.0
2,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos,0.0



Preferensi: Saya ingin mengolah sampah makanan menjadi sesuatu yang berguna.


,nama,kategori,deskripsi,skor_kemiripan
0,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos,0.525
1,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.362
2,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.141



Preferensi: Saya membutuhkan barang yang bisa dipakai berulang kali.


,nama,kategori,deskripsi,skor_kemiripan
0,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.461
1,Sedotan bambu,Perlengkapan,Sedotan ramah lingkungan yang bisa digunakan k...,0.310
2,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.058


### Pertanyaan refleksi
1. Apakah perubahan kalimat mengubah urutan rekomendasi?
2. Apakah produk dengan kata yang sama cenderung memperoleh skor lebih tinggi?
3. Apa yang terjadi ketika query menggunakan kata yang tidak ada di vocabulary?
4. Apakah TF-IDF memahami sinonim atau konteks?
5. Bandingkan hasilnya dengan notebook pretrained embeddings menggunakan data dan query yang sama.


## 10. Tampilkan semua skor
Tampilkan skor seluruh produk agar peserta dapat melihat bagaimana sistem menyusun ranking, bukan hanya Top-K.

In [10]:
def score_all_products(preference):
    query_tfidf = vectorizer.transform([preference])
    scores = cosine_similarity(query_tfidf, product_tfidf)[0]

    result = products.copy()
    result["skor_kemiripan"] = scores.round(3)
    return result.sort_values(
        "skor_kemiripan",
        ascending=False
    ).reset_index(drop=True)

score_all_products("Saya ingin mengurangi sampah plastik.")

,nama,kategori,deskripsi,skor_kemiripan
0,Botol minum reusable,Perlengkapan,Botol minum yang dapat digunakan berulang kali...,0.497
1,Kotak makan reusable,Perlengkapan,Wadah makanan yang dapat digunakan kembali unt...,0.342
2,Tas belanja kain,Perlengkapan,Tas kain yang dapat digunakan kembali saat ber...,0.173
3,Komposter rumah tangga,Pengelolaan sampah,Alat untuk mengolah sampah organik menjadi kompos,0.145
4,Lampu tenaga surya,Energi,Lampu hemat energi yang menggunakan tenaga mat...,0.000
5,Sedotan bambu,Perlengkapan,Sedotan ramah lingkungan yang bisa digunakan k...,0.000


## 11. Tantangan untuk peserta

**Tantangan A — Tambah item**
- Tambahkan minimal 3 produk baru dengan deskripsi yang spesifik.
- Jalankan ulang sel dataset.
- Jalankan ulang `fit_transform` pada katalog agar vocabulary dan matriks TF-IDF diperbarui.

**Tantangan B — Ubah Top-K**
- Bandingkan `top_k=1`, `top_k=3`, dan `top_k=5`.
- Apa yang berubah pada jumlah hasil dan relevansinya?

**Tantangan C — Uji keterbatasan lexical matching**
Coba query berikut:
- “Saya butuh wadah untuk bekal.”
- “Saya ingin barang yang ramah lingkungan.”
- “Saya ingin mengurangi penggunaan plastik.”

Bandingkan apakah produk yang masuk akal tetap mendapat skor tinggi jika kata-kata query tidak sama dengan deskripsi.

**Tantangan D — Perbaikan**
Usulkan cara meningkatkan sistem, misalnya menambah metadata, menggabungkan kolom nama dan deskripsi, menggunakan model embedding multilingual, atau mengevaluasi hasil dengan label relevansi.


## 12. Troubleshooting
- **Semua skor nol:** kemungkinan kata pada query tidak muncul di vocabulary produk, atau teks kosong setelah preprocessing.
- **Hasil kurang semantik:** TF-IDF mengandalkan overlap kata; sinonim dan konteks tidak dipahami secara otomatis.
- **Jumlah fitur sangat kecil:** katalog mungkin terlalu pendek atau deskripsinya terlalu mirip.
- **Menambah produk tetapi hasil tidak berubah:** jalankan ulang sel yang membangun `vectorizer` dan `product_tfidf`.
- **Skor bukan probabilitas:** skor cosine similarity menunjukkan kemiripan vektor, bukan peluang pengguna menyukai produk.

## 13. Ringkasan
Dalam hands-on ini kita membangun sistem rekomendasi sederhana dengan alur:

**Teks preferensi → TF-IDF vectorization → cosine similarity → ranking → Top-K rekomendasi**

TF-IDF adalah baseline yang mudah dijelaskan dan cepat dijalankan. Namun, pendekatan ini terutama mengandalkan overlap kata sehingga tidak secara langsung menangkap sinonim atau makna kontekstual seperti model embedding yang dirancang untuk semantic similarity.

### Pengembangan setelah kelas
1. Bandingkan hasil TF-IDF dengan pretrained embeddings pada **dataset, preferensi, dan Top-K yang sama**.
2. Gabungkan nama produk, kategori, dan deskripsi sebagai fitur teks.
3. Tambahkan filter kategori atau preferensi pengguna.
4. Evaluasi relevansi dengan contoh query dan label yang dinilai secara konsisten.
5. Bungkus fungsi rekomendasi dengan FastAPI.

**Referensi**
- `TfidfVectorizer`: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- Text feature extraction: https://scikit-learn.org/stable/modules/feature_extraction.html#text-feature-extraction
- Cosine similarity: https://scikit-learn.org/stable/modules/generated/sklearn.metrics.pairwise.cosine_similarity.html
